In [17]:
sealed class List<out A> {
    companion object {
        operator fun <A> invoke(vararg aa: A): List<A> =
            if (aa.isEmpty())
                Nil
            else
                Cons(aa[0], List(*aa.sliceArray(1 .. aa.size - 1)))

        fun <A> empty(): List<A> =
            Nil
    }
}

object Nil: List<Nothing>() {
    override fun toString(): String = "Nil"
}

data class Cons<out A>(val head: A, val tail: List<A>) : List<A>() {
    override fun toString(): String =
        "$head :: $tail"
}

fun <A, B> foldRight(xs: List<A>, z: B, f: (A, B) -> B): B =
    when (xs) {
        is Cons -> f(xs.head, foldRight(xs.tail, z, f))
        else -> z
    }

tailrec fun <A, B> foldLeft(xs: List<A>, z: B, f: (B, A) -> B): B =
    when (xs) {
        is Cons -> foldLeft(xs.tail, f(z, xs.head), f)
        else -> z
    }

fun <A, B> foldRightL(xs: List<A>, z: B, f: (A, B) -> B): B =
    foldLeft(xs, { b: B -> b }) { acc, a ->
        { b: B -> acc(f(a, b)) }
    }(z)

fun <A, B> List<A>.map(f: (A) -> B): List<B> =
    foldRightL(this, Nil as List<B>) { a, acc -> Cons(f(a), acc) }

println(List(1, 2, 3).map { it + 1 })

fun <A> length(xs: List<A>): Int =
    foldRightL(xs, 0) { _, l -> l + 1 }

fun List<Double>.sum(): Double =
    foldLeft(this, 0.0) { acc, n -> acc + n }

fun <A> List<A>.isEmpty(): Boolean = this is Nil


2 :: 3 :: 4 :: Nil


In [18]:
fun mean(xs: List<Double>): Double =
    if (xs.isEmpty())
        throw ArithmeticException("mean of empty list")
    else
        xs.sum() / length(xs)

println(mean(List(1.0, 2.0)))
//println(mean(List()))

1.5


In [19]:
sealed interface Option<out A>

fun <A> none(): Option<A> = None

data class Some<out A>(val get: A): Option<A> {
    override fun toString(): String = "Some($get)"
}

object None: Option<Nothing> {
    override fun toString(): String = "None"
}

In [20]:
fun mean(xs: List<Double>): Option<Double> =
    if (xs.isEmpty())
        None
    else
        Some(xs.sum() / length(xs))

println(mean(List(1.0, 2.0)))
println(mean(List()))

Some(1.5)
None


#### ex 4.1

In [21]:
fun <A, B> Option<A>.map(fn: (A) -> B): Option<B> =
    when (this) {
        is Some -> Some(fn(this.get))
        else -> None
    }

println(Some(4).map { it * 4 })
println(none<Int>().map { it * 4 })

fun <A, B> Option<A>.flatMap(fn: (A) -> Option<B>): Option<B> =
    when (this) {
        is Some -> fn(this.get)
        else -> None
    }

val divBy4: (Int) -> Option<Double> = { if (it == 0) None else Some(it / 4.0) }
println(Some(4).flatMap(divBy4))
println(Some(0).flatMap(divBy4))
println(None.flatMap(divBy4))

fun <A> Option<A>.getOrElse(default: () -> A): A =
    when (this) {
        is Some -> this.get
        else -> default()
    }

println(Some(4).getOrElse{ -1 })
println(None.getOrElse{ -1 })

fun <A> Option<A>.orElse(ob: () -> Option<A>): Option<A> =
    when (this) {
        is Some -> this
        else -> ob()
    }
println(Some(4).orElse{ Some(-1) })
println(None.orElse{ Some(-1) })

fun <A> Option<A>.filter(f: (A) -> Boolean): Option<A> =
    when (this) {
        is Some -> if (f(this.get)) this else None
        else -> None
    }

println(Some(4).filter { it % 2 == 0 })
println(Some(3).filter { it % 2 == 0 })
println(none<Int>().filter { it % 2 == 0 })


Some(16)
None
Some(1.0)
None
None
4
-1
Some(4)
Some(-1)
Some(4)
None
None


In [22]:
data class Employee(val name: String, val department: String, val manager: Option<String>)

fun lookupByName(name: String): Option<Employee> = TODO()

fun timDepartment(): Option<String> =
    lookupByName("Tim").map { it.department }

fun timManager(): String =
    lookupByName("Tim").flatMap { it.manager }.getOrElse { "Unemployed" }

In [23]:
fun variance(xs: List<Double>): Option<Double> =
    mean(xs).flatMap { m ->
        mean(xs.map { x ->
            Math.pow(x - m, 2.0)
        })
    }

println(variance(List(1.0, 2.0, 3.0)))
println(variance(List()))
println(variance(List(1.0)))

Some(0.6666666666666666)
None
Some(0.0)


In [24]:
fun <A, B> lift(f: (A) -> B): (Option<A>) -> Option<B> = { it.map(f) }

val absO: (Option<Double>) -> Option<Double> = lift(::abs)

println(absO(Some(-2.0)))

fun insuranceRateQuote(age: Int, numberOfSpeedingTickets: Int): Double = TODO()

Some(2.0)


In [25]:
fun <A> catches(a: () -> A): Option<A> =
    try { Some(a()) }
    catch (e: Exception) { None }


In [26]:
fun <A, B, C> map2(a: Option<A>, b: Option<B>, f: (A, B) -> C): Option<C> =
    a.flatMap { a -> b.map { b -> f(a, b) } }

fun parseInsuranceRateQuote(age: String, speedingTickets: String): Option<Double> =
    map2(catches { age.toInt() }, catches { speedingTickets.toInt() }, ::insuranceRateQuote)


#### ex 4.4

In [27]:
fun <A> sequence(xs: List<Option<A>>): Option<List<A>> =
    when (xs) {
        is Cons -> when(val h: Option<A> = xs.head) {
            is Some -> sequence(xs.tail).map { xs: List<A> -> Cons(h.get, xs) }
            else -> none<List<A>>()
        }
        else -> Some(List<A>())
    }

println(sequence(List(Some(1), None, Some(3))))
println(sequence(List(Some(1), Some(2), Some(3))))

fun <A> sequence2(xs: List<Option<A>>): Option<List<A>> =
    foldRight(xs, Some(List<A>())) { ox: Option<A>, acc: Option<List<A>> ->
        ox.flatMap { x -> acc.map { Cons(x, it) } }
    }

println(sequence2(List(Some(1), None, Some(3))))
println(sequence2(List(Some(1), Some(2), Some(3))))


None
Some(1 :: 2 :: 3 :: Nil)
None
Some(1 :: 2 :: 3 :: Nil)


#### ex 4.5

In [28]:
fun <A, B> traverse(xa: List<A>, f: (A) -> Option<B>): Option<List<B>> =
    foldRight(xa, Some(List<B>())) { a: A, acc: Option<List<B>> ->
        f(a).flatMap { b -> acc.map { Cons(b, it) } }
    }

val divBy2IfEven: (Int) -> Option<Int> = { if (it % 2 == 0) Some(it / 2) else None }
println(traverse(List(1, 2, 3), divBy2IfEven))
println(traverse(List(2, 4, 6), divBy2IfEven))

fun <A> sequence3(xs: List<Option<A>>): Option<List<A>> =
    traverse(xs) { it }

println(sequence3(List(Some(1), None, Some(3))))
println(sequence3(List(Some(1), Some(2), Some(3))))

None
Some(1 :: 2 :: 3 :: Nil)
None
Some(1 :: 2 :: 3 :: Nil)


In [29]:
sealed interface Either<out E, out A>

data class Left<out E>(val value: E): Either<E, Nothing> {
    override fun toString(): String = "Left($value)"
}

data class Right<out A>(val value: A): Either<Nothing, A> {
    override fun toString(): String = "Right($value)"
}

In [30]:
fun meanE(xs: List<Double>): Either<String, Double> =
    if (xs.isEmpty())
        Left("mean of empty list")
    else
        Right(xs.sum() / length(xs))

println(meanE(List()))
println(meanE(List(1.0, 2.0, 3.0)))

fun safeDiv(x: Int, y: Int): Either<Exception, Int> =
    try {
        Right(x / y)
    } catch (e: Exception) {
        Left(e)
    }

println(safeDiv(2, 0))
println(safeDiv(2, 1))

fun <A> catches(a: () -> A): Either<Exception, A> =
    try {
        Right(a())
    } catch (e: Exception) {
        Left(e)
    }

fun safeDiv2(x: Int, y: Int): Either<Exception, Int> =
    catches { x / y }

println(safeDiv2(2, 0))
println(safeDiv2(2, 1))


Left(mean of empty list)
Right(2.0)
Left(java.lang.ArithmeticException: / by zero)
Right(2)
Left(java.lang.ArithmeticException: / by zero)
Right(2)


#### 4.6

In [31]:
fun <E, A, B> Either<E, A>.map(f: (A) -> B): Either<E, B> =
    when(this) {
        is Right -> Right(f(this.value))
        is Left -> Left(this.value)
        else -> error("")
    }

println(safeDiv(2, 1).map { it * 10 })
println(safeDiv(2, 0).map { it * 10 })

fun <E, A, B> Either<E, A>.flatMap(f: (A) -> Either<E, B>): Either<E, B> =
    when(this) {
        is Right -> f(this.value)
        is Left -> Left(this.value)
        else -> error("")
    }

fun <E, A> Either<E, A>.orElse(f: () -> Either<E, A>): Either<E, A> =
    when(this) {
        is Right -> this
        is Left -> f()
        else -> error("")
    }

fun <E, A, B, C> map2(ae: Either<E, A>, be: Either<E, B>, f: (A, B) -> C): Either<E, C> =
    ae.flatMap { a ->
        be.map { b ->
            f(a, b)
        }
    }

println(map2(safeDiv(3, 1), safeDiv(2, 1)) { a, b -> a * b })
println(map2(safeDiv(3, 1), safeDiv(2, 0)) { a, b -> a * b })
println(map2(safeDiv(3, 0), safeDiv(2, 1)) { a, b -> a * b })

Right(20)
Left(java.lang.ArithmeticException: / by zero)
Right(6)
Left(java.lang.ArithmeticException: / by zero)
Left(java.lang.ArithmeticException: / by zero)


#### ex 4.7

In [32]:
fun <E, A, B> List<A>.traverse(f: (A) -> Either<E, B>): Either<E, List<B>> =
    foldLeft(this, Right(List())) { acc: Either<E, List<B>>, a: A ->
        acc.flatMap { bs ->
            f(a).map { b -> Cons(b, bs) }
        }
    }

println(List(1, 2, 3).traverse { if (it % 2 == 0) Right(it / 2) else Left("not even") })
println(List(2, 4, 6).traverse { if (it % 2 == 0) Right(it / 2) else Left("not even") })

fun <E, A> List<Either<E, A>>.sequence(): Either<E, List<A>> =
    this.traverse() { it }

println(List(Left("not even"), Right(2), Left("not even")).sequence())
println(List(Right(2), Right(4), Right(6)).sequence())


Left(not even)
Right(3 :: 2 :: 1 :: Nil)
Left(not even)
Right(6 :: 4 :: 2 :: Nil)


In [37]:
data class Name(val value: String)
data class Age(val value: Int)
data class Person(val name: Name, val age: Age)

fun mkName(name: String): Either<String, Name> =
    if (name.isBlank()) Left("Name is empty")
    else Right(Name(name))

fun mkAge(age: Int): Either<String, Age> =
    if (age < 0) Left("age is out of range")
    else Right(Age(age))

fun mkPerson(name: String, age: Int): Either<String, Person> =
    map2(mkName(name), mkAge(age)) { n, a -> Person(n, a) }

println(mkPerson("steve", 10))
println(mkPerson("steve", -10))
println(mkPerson("", 10))
println(mkPerson("", -10))

fun mkPerson2(name: String, age: Int): Either<List<String>, Person> {
    val en = mkName(name)
    val ea = mkAge(age)

    return when (en) {
        is Right -> when (ea) {
            is Right -> Right(Person(en.value, ea.value))
            is Left -> Left(List(ea.value))
            else -> error("")
        }

        is Left -> when (ea) {
            is Right -> Left(List(en.value))
            is Left -> Left(List(en.value, ea.value))
            else -> error("")
        }
        else -> error("")
    }
}

println(mkPerson2("steve", 10))
println(mkPerson2("steve", -10))
println(mkPerson2("", 10))
println(mkPerson2("", -10))

Right(Person(name=Name(value=steve), age=Age(value=10)))
Left(age is out of range)
Left(Name is empty)
Left(Name is empty)
Right(Person(name=Name(value=steve), age=Age(value=10)))
Left(age is out of range :: Nil)
Left(Name is empty :: Nil)
Left(Name is empty :: age is out of range :: Nil)
